# Análise de Índices de Vegetação

Pipeline de visão computacional para cálculo de VARI e ExG a partir de imagens RGB.

## Etapa 1 — Carregar imagem e converter BGR → RGB

In [ ]:
import matplotlib.pyplot as plt
import cv2
import numpy as np

# ── Imagem principal do projeto: imagem aérea de drone de ALTA RESOLUÇÃO ──────
# Obtida de banco de imagens aberto na internet (origem documentada no relatório),
# conforme permitido na proposta (seção 5.1 — download documentado).
# Escolhida por: alta resolução (19.9 MP), visada vertical (nadir) e variedade de
# superfícies (campo, estrada de terra, mata e céu) — ideal para histograma e segmentação.
img_bgr  = cv2.imread('../images/drone-campo-fazenda.jpg')

# OpenCV carrega em BGR; Matplotlib exibe em RGB — a conversão corrige a ordem dos canais
img_orig = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

## Etapa 2 — Inspecionar shape, dtype e pixel

In [ ]:
import os

caminho = '../images/drone-campo-fazenda.jpg'

# shape retorna (altura, largura, canais) — eixo 0 é vertical, eixo 1 horizontal
altura, largura, canais = img_orig.shape
print(f'Resolucao:        {largura} x {altura} pixels  ({largura*altura/1e6:.1f} MP)')
print(f'Numero de canais: {canais}  (R, G, B)')

# dtype indica o tipo de cada elemento; uint8 = inteiro sem sinal de 0 a 255
print(f'Dtype:            {img_orig.dtype}  (inteiros 0-255)')

# Tamanho do arquivo em disco, em KB
print(f'Tamanho em disco: {os.path.getsize(caminho)/1024:.1f} KB')

# Valor medio de pixel por canal — metrica pedida na proposta
medio_rgb = img_orig.reshape(-1, 3).mean(axis=0)
print(f'Valor medio (R,G,B): [{medio_rgb[0]:.1f}, {medio_rgb[1]:.1f}, {medio_rgb[2]:.1f}]  '
      f'(G dominante = cena verde)')

# img[y, x] acessa o pixel na linha y, coluna x — retorna [R, G, B]
pixel_y, pixel_x = 100, 200
print(f'Pixel [{pixel_y}, {pixel_x}]:   {img_orig[pixel_y, pixel_x]}')

## Etapa 2.1 — Redimensionamento (versão de trabalho)

A imagem de drone tem 19.9 MP — pesada para os filtros e geraria arquivos de saída enormes.
O **redimensionamento** com `cv2.resize` cria uma versão proporcional menor. Usamos a
interpolação `INTER_AREA`, recomendada para *reduzir* imagens (faz média das regiões, evita
serrilhado). **Toda a análise a seguir roda sobre esta versão de trabalho (`img_rgb`)**, enquanto
os metadados de resolução acima referem-se à imagem original em alta resolução.

In [ ]:
# Imagem de 19.9 MP é pesada para os filtros e geraria PNGs enormes.
# Redimensiona para 1600 px de largura mantendo a proporção (escala única em x e y).
# IMPORTANTE: toda a análise a seguir usa 'img_rgb' (a versão de trabalho).
largura_trabalho = 1600
escala = largura_trabalho / largura

# INTER_AREA: melhor para reducao — pondera a area dos pixels de origem
img_rgb = cv2.resize(img_orig, (largura_trabalho, int(altura * escala)),
                     interpolation=cv2.INTER_AREA)

print(f'Original (metadados):  {largura} x {altura} pixels  ({largura*altura/1e6:.1f} MP)')
print(f'Versao de trabalho:    {img_rgb.shape[1]} x {img_rgb.shape[0]} pixels  '
      f'({img_rgb.shape[0]*img_rgb.shape[1]/1e6:.1f} MP)')
print(f'Reducao:               {100*(1 - img_rgb.size/img_orig.size):.1f}% menos dados')

## Etapa 2.2 — Conversão para escala de cinza (original × processado)

Comparação visual pedida na proposta: a versão em escala de cinza preserva o **brilho**
mas descarta a **cor**. Repare que o campo verde e a mata ficam difíceis de distinguir
do solo em cinza — evidência de que **brilho não identifica vegetação**, motivando os
índices espectrais.

In [ ]:
# Conversão para escala de cinza — média ponderada perceptual (ITU-R BT.601):
# 0.299R + 0.587G + 0.114B — o olho humano é mais sensível ao verde
gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
axes[0].imshow(img_rgb)
axes[0].set_title(f'Original RGB  ({img_rgb.shape[1]}×{img_rgb.shape[0]})', fontsize=12)
axes[1].imshow(gray, cmap='gray')
axes[1].set_title('Escala de cinza (BT.601)', fontsize=12)
for a in axes:
    a.axis('off')
plt.tight_layout()
plt.savefig('../outputs/original_vs_grayscale.png', dpi=130, bbox_inches='tight')
plt.show()

print(f'Grayscale — min: {gray.min()}, max: {gray.max()}, média: {gray.mean():.1f}')
print('Figura salva em outputs/original_vs_grayscale.png')
# Observação-chave: na versão cinza, campo verde e mata escura ficam parecidos —
# a conversão descarta a COR, que é justamente o que define vegetação.
# Isso motiva os índices espectrais das etapas 4 e 5.

## Etapa 3 — Separar canais R, G, B

In [ ]:
# Converte para float32 — necessário antes de dividir para não truncar decimais
img_f = img_rgb.astype(np.float32)

# Normaliza para [0, 1] dividindo por 255.0
# VARI foi projetado para refletância (0–1); sem isso o numerador pode chegar a 255,
# fazendo outliers explodirem até 255× mais quando o denominador é próximo de zero
img_f = img_f / 255.0

# Fatia o eixo 2: [:, :, n] = todas as linhas, todas as colunas, canal n
R = img_f[:, :, 0]  # vermelho
G = img_f[:, :, 1]  # verde
B = img_f[:, :, 2]  # azul

print('Shape de cada canal:', R.shape)
print(f'R — min: {R.min():.4f}, max: {R.max():.4f}')
print(f'G — min: {G.min():.4f}, max: {G.max():.4f}')
print(f'B — min: {B.min():.4f}, max: {B.max():.4f}')

## Etapa 4 — Calcular VARI

In [ ]:
# Numerador: pixels com G > R são positivos (vegetação); G < R são negativos (solo, estruturas)
# Denominador: com valores em [0,1] o epsilon 1e-6 tem peso proporcional real
vari_raw = (G - R) / (G + R - B + 1e-6)

print('--- Antes do clip ---')
print(f'VARI raw — min: {vari_raw.min():.2f}, max: {vari_raw.max():.2f}')
print(f'Pixels com vari_raw < -1: {(vari_raw < -1).sum()}')
print(f'Pixels com vari_raw >  1: {(vari_raw >  1).sum()}')

# Limita ao intervalo [-1, 1] — outliers de pixels saturados não distorcem o colormap
vari = np.clip(vari_raw, -1, 1)

print('\n--- Após o clip ---')
print(f'VARI — min: {vari.min():.6f}, max: {vari.max():.6f}, média: {vari.mean():.4f}')
print(f'Pixels com VARI > 0: {(vari > 0).sum()}')
print(f'Pixels com VARI < 0: {(vari < 0).sum()}')

## Etapa 4.1 — Máscara de luminância

In [ ]:
# Brilho médio por pixel — proxy de luminância com valores em [0, 1]
luminance = (R + G + B) / 3

# Pixels abaixo do threshold são sombras onde G+R-B≈0 não tem significado físico de vegetação
threshold = 0.1
mask_sombra = luminance < threshold

# Copia o VARI e zera as sombras — np.nan seria alternativa, mas 0 é mais seguro para colormap
vari_masked = vari.copy()
vari_masked[mask_sombra] = 0

total_pixels = mask_sombra.size
print(f'Pixels mascarados (sombra):   {mask_sombra.sum():>8} ({100 * mask_sombra.mean():.2f}%)')
print(f'Pixels válidos para análise:  {(~mask_sombra).sum():>8} ({100 * (~mask_sombra).mean():.2f}%)')
print()
print(f'VARI médio — com sombras:     {vari.mean():.4f}')
print(f'VARI médio — sem sombras:     {vari_masked[~mask_sombra].mean():.4f}')

## Etapa 5 — Calcular ExG

In [ ]:
# Com valores normalizados em [0,1], o range teórico do ExG passa a ser [-2, +2]
# Valores positivos = verde dominante (vegetação); negativos = vermelho/azul dominante
exg = 2 * G - R - B

print('ExG shape:', exg.shape)
print(f'ExG — min: {exg.min():.4f}, max: {exg.max():.4f}, média: {exg.mean():.4f}')
print(f'\nComparação de médias:')
print(f'  VARI (normalizado -1 a 1): {vari.mean():.4f}')
print(f'  ExG  (normalizado -2 a 2): {exg.mean():.4f}')

## Etapa 6 — Visualização comparativa

## Etapa 7 — Segmentação binária por ExG

In [ ]:
# ExG > 0 = excesso de verde sobre vermelho+azul → vegetação
# Zero é o corte natural da fórmula com valores normalizados em [-2, +2]
threshold_exg = 0.0
mascara_veg = exg > threshold_exg

# Converte para uint8 (0 ou 255) para visualização e salvamento via OpenCV
mascara_bin = mascara_veg.astype(np.uint8) * 255

pct_veg = mascara_veg.mean() * 100
print(f'Threshold ExG: {threshold_exg}')
print(f'Pixels classificados como vegetação: {mascara_veg.sum():>8} ({pct_veg:.2f}%)')
print(f'Pixels classificados como não-veg:   {(~mascara_veg).sum():>8} ({100 - pct_veg:.2f}%)')


def destacar(img_rgb, mask_bool, cor_contorno=(255, 120, 50), escurecer=0.55, espessura=2):
    """Destaque por dessaturação: a região detectada MANTÉM a cor original;
    o resto da cena vira cinza escurecido. O contorno marca o limite.
    Mostra a detecção sem pintar nada por cima — nenhuma informação é coberta."""
    g3 = cv2.cvtColor(cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY), cv2.COLOR_GRAY2RGB)
    out = np.where(mask_bool[..., None], img_rgb, (g3 * escurecer).astype(np.uint8))
    cont, _ = cv2.findContours(mask_bool.astype(np.uint8),
                               cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(out, cont, -1, cor_contorno, espessura)
    return out


fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(img_rgb)
axes[0].set_title('Imagem original (RGB)', fontsize=13)
axes[0].axis('off')

axes[1].imshow(mascara_bin, cmap='gray')
axes[1].set_title(f'Segmentação ExG > {threshold_exg}  (branco = vegetação)', fontsize=13)
axes[1].axis('off')

axes[2].imshow(destacar(img_rgb, mascara_veg))
axes[2].set_title('Destaque — vegetação em cor, resto em cinza', fontsize=13)
axes[2].axis('off')

plt.tight_layout()
plt.savefig('../outputs/segmentacao_exg.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/segmentacao_exg.png')

## Etapa 8 — Análise comparativa entre as duas imagens

Aplica o índice ExG (baseline) às duas imagens do projeto, que representam dois
**regimes de cena** complementares:

- `drone-campo-fazenda` — campo misto com estrada e céu (a imagem principal);
- `drone-lavoura-solo` — lavoura com solo exposto (o caso difícil).

A comparação ExG × VARI por cena revela onde o baseline erra — na `lavoura-solo` o ExG
diz 97% de vegetação mas o VARI médio é negativo, denunciando o falso positivo que a
segmentação aprimorada (Etapa 13) corrige.

In [ ]:
from pathlib import Path

def processar_imagem(path, threshold_exg=0.0, lum_threshold=0.1):
    img_bgr = cv2.imread(str(path))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_f   = img_rgb.astype(np.float32) / 255.0

    R, G, B = img_f[:,:,0], img_f[:,:,1], img_f[:,:,2]

    # VARI com máscara de luminância
    vari_raw = (G - R) / (G + R - B + 1e-6)
    vari     = np.clip(vari_raw, -1, 1)
    mask_sombra = (R + G + B) / 3 < lum_threshold
    vari[mask_sombra] = 0

    # ExG e segmentação binária
    exg        = 2*G - R - B
    mascara_veg = exg > threshold_exg

    return {
        'nome':       path.stem,
        'img_rgb':    img_rgb,
        'vari':       vari,
        'exg':        exg,
        'mascara':    mascara_veg,
        'destaque':   destacar(img_rgb, mascara_veg, espessura=3),
        'pct_veg':    mascara_veg.mean() * 100,
        'vari_medio': vari[~mask_sombra].mean(),
    }

# As duas imagens de drone do projeto (a NASA entra só na Etapa 14)
imagens = [Path('../images/drone-campo-fazenda.jpg'),
           Path('../images/drone-lavoura-solo.jpg')]

resultados = [processar_imagem(p) for p in imagens]

# --- Tabela resumo ---
print(f'{"Imagem":<45} {"Veg %":>7} {"VARI médio":>12}')
print('-' * 66)
for r in resultados:
    print(f'{r["nome"]:<45} {r["pct_veg"]:>6.2f}% {r["vari_medio"]:>12.4f}')

# --- Grade visual: 1 linha por imagem, 3 colunas ---
fig, axes = plt.subplots(len(resultados), 3, figsize=(18, 6 * len(resultados)))

for i, r in enumerate(resultados):
    axes[i, 0].imshow(r['img_rgb'])
    axes[i, 0].set_title(f'{r["nome"]}\nOriginal', fontsize=11)
    axes[i, 0].axis('off')

    # Exibição em ±0.4 (realce visual; os DADOS e estatísticas usam o range completo)
    im = axes[i, 1].imshow(r['vari'], cmap='RdYlGn', vmin=-0.4, vmax=0.4)
    axes[i, 1].set_title(f'VARI  (médio: {r["vari_medio"]:.3f}, exibição ±0.4)', fontsize=11)
    axes[i, 1].axis('off')
    plt.colorbar(im, ax=axes[i, 1], fraction=0.046, pad=0.04)

    axes[i, 2].imshow(r['destaque'])
    axes[i, 2].set_title(f'ExG — detecção destacada  ({r["pct_veg"]:.1f}% vegetação)', fontsize=11)
    axes[i, 2].axis('off')

plt.tight_layout()
plt.savefig('../outputs/analise_multi_imagem.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/analise_multi_imagem.png')

## Etapa 9a — Histograma do ExG

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

# ravel() transforma (H, W) em vetor 1D sem copiar dados
ax.hist(exg.ravel(), bins=300, color='steelblue', alpha=0.75, label='ExG')

# Marca o threshold fixo atual para comparação visual com o futuro threshold Otsu
ax.axvline(0.0, color='tomato', linestyle='--', linewidth=1.5, label='threshold fixo = 0.0')

ax.set_xlabel('ExG  (2G − R − B),  valores normalizados [−2, +2]', fontsize=11)
ax.set_ylabel('Número de pixels', fontsize=11)
ax.set_title('Distribuição do ExG — drone-campo-fazenda', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/histograma_exg.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/histograma_exg.png')

## Etapa 9b — Histograma dos canais R, G e B

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))

# ravel() achata (H, W) em vetor 1D sem copiar dados
for canal, cor, nome in zip([R, G, B], ['tomato', 'seagreen', 'steelblue'], ['R', 'G', 'B']):
    ax.hist(canal.ravel(), bins=256, color=cor, alpha=0.55, label=nome)

ax.set_xlabel('Intensidade [0, 1]', fontsize=11)
ax.set_ylabel('Número de pixels', fontsize=11)
ax.set_title('Histograma dos canais R, G, B — drone-campo-fazenda', fontsize=13)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/histograma_rgb.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/histograma_rgb.png')

## Etapa 9c — Equalização de histograma (realce de contraste)

A **equalização de histograma** redistribui as intensidades para ocupar toda a faixa de tons,
realçando o contraste **global** — técnica de melhoria visual prevista na proposta. Equalizamos
apenas a **luminância** (canal Y do espaço YCrCb), preservando as cores. O desvio-padrão do
brilho aumenta: sinal de que os tons ficaram mais espalhados.

> **Conexão com o resto do projeto:** a equalização é um realce *global* (uma única curva para
> a imagem inteira). Seu parente **local** — o *white top-hat* — reaparece na **Etapa 15** para
> revelar estrelas fracas, ajustando o contraste vizinhança a vizinhança. Mesma família de
> ferramentas morfológicas, em escalas diferentes.

In [ ]:
# Equalização de histograma — realce de contraste GLOBAL (técnica da tabela 4.3 da proposta).
# Equaliza só a LUMINÂNCIA (canal Y do YCrCb), preservando a cor — equalizar R, G, B
# separadamente distorceria os matizes.
ycrcb = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2YCrCb)
ycrcb[:, :, 0] = cv2.equalizeHist(ycrcb[:, :, 0])
img_eq = cv2.cvtColor(ycrcb, cv2.COLOR_YCrCb2RGB)
gray_eq = cv2.cvtColor(img_eq, cv2.COLOR_RGB2GRAY)

print(f'Desvio-padrao do brilho — original: {gray.std():.1f}  ->  equalizado: {gray_eq.std():.1f}')
print('Maior desvio-padrao = tons mais espalhados = mais contraste.')

fig, ax = plt.subplots(1, 3, figsize=(20, 5))
ax[0].imshow(img_rgb);  ax[0].set_title('Original', fontsize=12);  ax[0].axis('off')
ax[1].imshow(img_eq);   ax[1].set_title('Equalizada (realce de contraste)', fontsize=12); ax[1].axis('off')
ax[2].hist(gray.ravel(),    bins=256, color='gray',   alpha=0.6, label='original')
ax[2].hist(gray_eq.ravel(), bins=256, color='tomato', alpha=0.5, label='equalizado')
ax[2].set_title('Histograma de brilho: antes × depois', fontsize=12)
ax[2].set_yticks([]); ax[2].legend(fontsize=10)
plt.tight_layout()
plt.savefig('../outputs/equalizacao_histograma.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/equalizacao_histograma.png')

## Etapa 10 — Comparação: Otsu (grayscale) vs. ExG threshold fixo

In [ ]:
# ── Kernel morfológico compartilhado pelos dois métodos ──────────────────────
# 7×7 pixels: remove ruídos pequenos e preenche buracos sem deformar regiões grandes
kernel = np.ones((7, 7), np.uint8)

def refinar_mascara(mask_bin):
    """Opening (erosão→dilatação) remove ilhas de ruído;
       Closing (dilatação→erosão) preenche buracos internos."""
    aberta   = cv2.morphologyEx(mask_bin, cv2.MORPH_OPEN,  kernel, iterations=2)
    fechada  = cv2.morphologyEx(aberta,   cv2.MORPH_CLOSE, kernel, iterations=2)
    return fechada

def extrair_contornos(mask_refinada, img_base, cor_rgb):
    """Encontra contornos externos e os desenha sobre uma cópia da imagem."""
    contornos, _ = cv2.findContours(mask_refinada, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    overlay = img_base.copy()
    cv2.drawContours(overlay, contornos, -1, cor_rgb, thickness=4)
    return overlay, len(contornos)

# ── Método 1: Otsu sobre imagem em escala de cinza ───────────────────────────
# Otsu segmenta por intensidade de brilho — threshold automático pelo histograma
gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
thresh_otsu, mask_otsu_bruta = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
mask_otsu_ref = refinar_mascara(mask_otsu_bruta)
contornos_otsu, n_otsu = extrair_contornos(mask_otsu_ref, img_rgb, cor_rgb=(255, 165, 0))   # laranja
pct_otsu = (mask_otsu_ref > 0).mean() * 100

# ── Método 2: Threshold fixo ExG > 0 ─────────────────────────────────────────
# ExG segmenta pelo excesso de verde sobre vermelho+azul — threshold de significado físico
mask_exg_bruta = (exg > 0).astype(np.uint8) * 255
mask_exg_ref   = refinar_mascara(mask_exg_bruta)
contornos_exg, n_exg = extrair_contornos(mask_exg_ref, img_rgb, cor_rgb=(0, 230, 80))       # verde
pct_exg = (mask_exg_ref > 0).mean() * 100

# ── Resumo numérico ───────────────────────────────────────────────────────────
print(f'{"Método":<30} {"Threshold":>12} {"Cobertura":>10} {"Regiões":>8}')
print('-' * 64)
print(f'{"Otsu (grayscale)":<30} {thresh_otsu/255:>12.4f} {pct_otsu:>9.2f}% {n_otsu:>8}')
print(f'{"ExG fixo (> 0)":<30} {"0.0000":>12} {pct_exg:>9.2f}% {n_exg:>8}')

# ── Visualização 2×3 ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
titulos_col = ['Máscara bruta', 'Máscara refinada\n(opening + closing)', 'Contornos sobre original']

for col, titulo in enumerate(titulos_col):
    axes[0, col].set_title(f'Otsu grayscale — {titulo}', fontsize=11)
    axes[1, col].set_title(f'ExG > 0 — {titulo}', fontsize=11)

axes[0, 0].imshow(mask_otsu_bruta, cmap='gray');  axes[0, 0].axis('off')
axes[0, 1].imshow(mask_otsu_ref,   cmap='gray');  axes[0, 1].axis('off')
axes[0, 2].imshow(contornos_otsu);                axes[0, 2].axis('off')
axes[0, 2].set_title(f'Otsu — contornos (laranja)  |  {pct_otsu:.1f}% cobertura  |  {n_otsu} regiões', fontsize=11)

axes[1, 0].imshow(mask_exg_bruta, cmap='gray');   axes[1, 0].axis('off')
axes[1, 1].imshow(mask_exg_ref,   cmap='gray');   axes[1, 1].axis('off')
axes[1, 2].imshow(contornos_exg);                 axes[1, 2].axis('off')
axes[1, 2].set_title(f'ExG > 0 — contornos (verde)  |  {pct_exg:.1f}% cobertura  |  {n_exg} regiões', fontsize=11)

plt.tight_layout()
plt.savefig('../outputs/comparacao_segmentacao.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/comparacao_segmentacao.png')

## Etapa 11 — Filtragem comparativa: Gaussiano, Mediana, Bilateral

### Filtro 1 — Gaussiano (σ = 1.5)

In [ ]:
import time

# img_f já está normalizada em [0, 1] float32 — base comum para os três filtros
# ksize=(0,0) instrui o OpenCV a calcular o tamanho do kernel a partir do sigma
# sigmaX=1.5: raio de influência suave — borra ruído de alta frequência sem destruir bordas
t0 = time.time()
gaussiano = cv2.GaussianBlur(img_f, ksize=(0, 0), sigmaX=1.5)
t_gauss = time.time() - t0

print(f'Gaussiano  σ=1.5  →  tempo: {t_gauss*1000:.1f} ms')

### Filtro 2 — Mediana (kernel 5×5)

In [ ]:
# medianBlur requer float32 com ksize ≤ 5 — converte para uint8, aplica, volta para float
# Alternativa: usar img_rgb uint8 diretamente e normalizar só no PSNR
t0 = time.time()
img_u8 = (img_f * 255).astype(np.uint8)
mediana_u8 = cv2.medianBlur(img_u8, ksize=5)
mediana = mediana_u8.astype(np.float32) / 255.0   # volta para [0,1] para comparação uniforme
t_median = time.time() - t0

print(f'Mediana    5×5    →  tempo: {t_median*1000:.1f} ms')

### Filtro 3 — Bilateral (d=9, σColor=75, σSpace=75)

In [ ]:
# bilateralFilter também trabalha melhor em uint8 para os sigma em escala 0-255
# sigmaColor=75: pixels com Δintensidade > 75 não são mesclados → borda preservada
# sigmaSpace=75: raio espacial do Gaussiano de proximidade
t0 = time.time()
bilateral_u8 = cv2.bilateralFilter(img_u8, d=9, sigmaColor=75, sigmaSpace=75)
bilateral = bilateral_u8.astype(np.float32) / 255.0
t_bilateral = time.time() - t0

print(f'Bilateral  d=9    →  tempo: {t_bilateral*1000:.1f} ms')
print()
print(f'Custo relativo ao Gaussiano:')
print(f'  Mediana:   {t_median/t_gauss:>5.1f}×')
print(f'  Bilateral: {t_bilateral/t_gauss:>5.1f}×')

### PSNR e visualização comparativa

In [ ]:
def psnr(original, filtrada):
    """PSNR em dB para imagens normalizadas em [0, 1].
    MAX=1.0; quanto maior o valor, mais próximo da original."""
    mse = np.mean((original.astype(np.float64) - filtrada.astype(np.float64)) ** 2)
    return 20 * np.log10(1.0 / np.sqrt(mse)) if mse > 0 else float('inf')

def snr(original, filtrada):
    """SNR em dB: potência do sinal / potência do ruído (ruído = diferença)."""
    o = original.astype(np.float64)
    ruido = o - filtrada.astype(np.float64)
    pot_sinal = np.mean(o ** 2)
    pot_ruido = np.mean(ruido ** 2)
    return 10 * np.log10(pot_sinal / pot_ruido) if pot_ruido > 0 else float('inf')

filtros = [
    ('Original',           img_f,      None),
    (f'Gaussiano σ=1.5\n{t_gauss*1000:.0f} ms',    gaussiano,  t_gauss),
    (f'Mediana 5×5\n{t_median*1000:.0f} ms',        mediana,    t_median),
    (f'Bilateral d=9\n{t_bilateral*1000:.0f} ms',   bilateral,  t_bilateral),
]

print(f'{"Filtro":<20} {"PSNR (dB)":>10} {"SNR (dB)":>10} {"Tempo (ms)":>12}')
print('-' * 56)
for nome, img_fil, tempo in filtros[1:]:
    p = psnr(img_f, img_fil)
    s = snr(img_f, img_fil)
    print(f'{nome.split(chr(10))[0]:<20} {p:>10.2f} {s:>10.2f} {tempo*1000:>11.1f}')

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
for ax, (nome, img_fil, tempo) in zip(axes, filtros):
    ax.imshow(np.clip(img_fil, 0, 1))
    titulo = nome
    if tempo is not None:
        titulo += f'\nPSNR: {psnr(img_f, img_fil):.2f} dB | SNR: {snr(img_f, img_fil):.2f} dB'
    ax.set_title(titulo, fontsize=11)
    ax.axis('off')

plt.suptitle('Comparação de filtros — drone-campo-fazenda', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/comparacao_filtros.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/comparacao_filtros.png')

## Etapa 12a — Detecção de bordas (Canny): discussão dos limiares

O operador de Canny usa **histerese** com dois limiares (t1/t2). A escolha controla o
compromisso:

- **Limiares baixos (30/90)** — capturam até a textura fina da plantação → muitas bordas "ruidosas";
- **Limiares médios (50/130)** — equilíbrio: estrada, horizonte e bordas de talhões ficam contínuos (par adotado no projeto, próximo do 50/150 sugerido na proposta);
- **Limiares altos (150/250)** — só os contornos mais fortes sobrevivem → bordas fragmentadas.

O resultado do Canny é, por definição, **fundo preto com linhas brancas** (preto = "sem borda").

In [ ]:
# O Canny exige escala de cinza pré-suavizada — o Gaussiano reduz bordas espúrias de textura
gray_gauss = cv2.cvtColor((np.clip(gaussiano, 0, 1) * 255).astype(np.uint8),
                           cv2.COLOR_RGB2GRAY)

# Histerese com dois limiares: gradiente > t2 = borda forte (aceita);
# entre t1 e t2 = borda fraca (aceita SÓ se conectada a uma forte); < t1 = descarta
pares = [(30, 90), (50, 130), (150, 250)]

fig, axes = plt.subplots(1, len(pares) + 1, figsize=(22, 5))
axes[0].imshow(img_rgb)
axes[0].set_title('Original', fontsize=11)
axes[0].axis('off')

print(f'{"Limiares (t1/t2)":<18} {"% pixels de borda":>18}')
print('-' * 38)
for ax, (t1, t2) in zip(axes[1:], pares):
    edges = cv2.Canny(gray_gauss, threshold1=t1, threshold2=t2)
    pct = (edges > 0).mean() * 100
    ax.imshow(edges, cmap='gray')
    ax.set_title(f'Canny {t1}/{t2}  ({pct:.2f}% bordas)', fontsize=11)
    ax.axis('off')
    print(f'{t1}/{t2:<14} {pct:>17.2f}%')

plt.suptitle('Detecção de bordas (Canny) — efeito dos limiares de histerese', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/canny_limiares.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/canny_limiares.png')

## Etapa 12b — Grid comparativo final

Consolida todos os estágios do pipeline em um único painel 2×5, como pede a proposta
("grid comparativo com todas as etapas via matplotlib").

In [ ]:
# ── Canny no par adotado (50/130), sobre o gray_gauss da Etapa 12a ───────────
canny = cv2.Canny(gray_gauss, threshold1=50, threshold2=130)

# ── Contornos ExG sobre a imagem original (máscara refinada da etapa 10) ──────
overlay_final = img_rgb.copy()
contornos_f, _ = cv2.findContours(mask_exg_ref, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(overlay_final, contornos_f, -1, (0, 230, 80), thickness=4)

# ── Grid 2×5 ─────────────────────────────────────────────────────────────────
paineis = [
    (img_rgb,           'Original',                          'gray'),
    (gray,              'Escala de cinza',                   'gray'),
    (gaussiano,         'Gaussiano  σ=1.5',                  None),
    (mediana,           'Mediana  5×5',                      None),
    (bilateral,         'Bilateral  d=9',                    None),
    (canny,             'Canny  (bordas)',                   'gray'),
    (vari_masked,       'VARI  (exibição ±0.4)',             'RdYlGn'),
    (exg,               'ExG  (simétrico, amarelo = 0)',     'exg'),
    (mask_otsu_ref,     'Otsu + morfologia',                 'gray'),
    (overlay_final,     'ExG > 0 + contornos',               None),
]

fig, axes = plt.subplots(2, 5, figsize=(26, 10))
axes_flat = axes.flatten()

for ax, (img_data, titulo, cmap) in zip(axes_flat, paineis):
    if cmap == 'RdYlGn':
        # Exibição em ±0.4 — realce visual; dados intactos
        ax.imshow(img_data, cmap=cmap, vmin=-0.4, vmax=0.4)
    elif cmap == 'exg':
        # Colormap divergente centrado no limiar físico 0 (amarelo = limite veg/não-veg)
        m = np.percentile(np.abs(img_data), 98)
        ax.imshow(img_data, cmap='RdYlGn', vmin=-m, vmax=m)
    elif cmap == 'gray':
        ax.imshow(img_data, cmap='gray')
    else:
        # overlay_final é uint8 (0-255); os filtros são float [0,1].
        # Clipar um uint8 em [0,1] zeraria a imagem (ficaria preta) — trata cada caso.
        ax.imshow(img_data if img_data.dtype == np.uint8 else np.clip(img_data, 0, 1))
    ax.set_title(titulo, fontsize=10, pad=6)
    ax.axis('off')

fig.suptitle(
    'Pipeline completo — Análise de Vegetação por Índices Espectrais RGB',
    fontsize=14, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.savefig('../outputs/grid_pipeline_completo.png', dpi=200, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/grid_pipeline_completo.png')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Imagem original ---
axes[0].imshow(img_rgb)
axes[0].set_title('Imagem original (RGB)', fontsize=13)
axes[0].axis('off')

# --- Mapa VARI com máscara de luminância ---
# Exibição em ±0.4: o range físico [-1,1] deixaria o mapa pálido (média 0.14).
# Realce apenas visual — os dados e estatísticas usam o range completo.
im_vari = axes[1].imshow(vari_masked, cmap='RdYlGn', vmin=-0.4, vmax=0.4)
axes[1].set_title(f'VARI  (máscara luminância < {threshold}, exibição ±0.4)', fontsize=13)
axes[1].axis('off')
plt.colorbar(im_vari, ax=axes[1], fraction=0.046, pad=0.04)

# --- Mapa ExG ---
# Colormap divergente DEVE ser centrado no limiar de decisão (0):
# range simétrico ±percentil 98 garante que amarelo = ExG 0 (limite vegetação/não-vegetação)
m98 = np.percentile(np.abs(exg), 98)
im_exg = axes[2].imshow(exg, cmap='RdYlGn', vmin=-m98, vmax=m98)
axes[2].set_title(f'ExG  (simétrico ±{m98:.2f}, amarelo = limiar 0)', fontsize=13)
axes[2].axis('off')
plt.colorbar(im_exg, ax=axes[2], fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig('../outputs/comparativo_indices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/comparativo_indices.png')

## Etapa 13 — Segmentação aprimorada: testes absolutos de cor

A segmentação `ExG > 0` da Etapa 7 é ingênua: classifica como vegetação qualquer pixel onde o
verde é *minimamente* maior que a média de R e B. Isso gera **falsos positivos** em céu, solo claro,
cinza e reflexos.

Mas há um segundo erro a evitar: usar **Otsu** (um corte relativo) faz o oposto — em cenas quase
100% verdes, o Otsu corta *dentro* da vegetação real, gerando **falsos negativos**. A solução robusta
usa **testes absolutos de cor**, que não dependem da proporção verde/fundo da cena:

| # | Técnica | O que resolve |
|---|---|---|
| 1 | **ExGR = 3G − 2.4R − B** (Meyer & Neto, 2008) | Subtrai o excesso de vermelho; separa solo de vegetação |
| 2 | **Trava HSV** `H∈[30,95] & S≥40 & V≥30` | Rejeita cinza/dessaturado — corrige os falsos positivos |
| 3 | **Preenchimento de buracos** | Fecha vazios internos nas regiões de vegetação |
| 4 | **Filtro de área mínima** | Remove regiões minúsculas antes de desenhar contornos |

O pixel só conta como vegetação se **ExGR > 0** (índice confirma verde dominante) **E** passar na
trava HSV (cor realmente verde e saturada). Os dois testes são absolutos — robustos a cenas
uniformes, ao contrário do Otsu.

In [ ]:
from scipy import ndimage

def segmentar_aprimorado(img_rgb, min_area_frac=0.0005):
    """Segmentação de vegetação de alta precisão por testes absolutos de cor.

    Combina o índice ExGR (Meyer & Neto) com uma trava de cor em HSV, refino
    morfológico e filtragem por área mínima. Não usa Otsu: testes absolutos evitam
    tanto falsos positivos (cinza/solo) quanto falsos negativos (cortar vegetação).

    Retorna a máscara binária de vegetação.
    """
    f = img_rgb.astype(np.float32) / 255.0
    R, G, B = f[:, :, 0], f[:, :, 1], f[:, :, 2]

    # 1. ExGR = ExG - ExR = 3G - 2.4R - B  → positivo onde há verde dominante
    exgr = 3*G - 2.4*R - B

    # 2. Trava HSV: precisa ser verde de matiz, saturado e não-sombra
    hsv = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2HSV)
    H, S, V = hsv[:, :, 0], hsv[:, :, 1], hsv[:, :, 2]
    gate = (H >= 30) & (H <= 95) & (S >= 40) & (V >= 30)

    # Combinação por AND — ambos os testes são absolutos
    mask = ((exgr > 0) & gate).astype(np.uint8)

    # 3. Morfologia — FECHAMENTO PRIMEIRO: a plantação aparece como fileiras finas de
    #    verde separadas por sulcos de terra. Abrir primeiro (versão antiga) apagava
    #    essas fileiras. Fechar primeiro UNE as fileiras num bloco de cultura; só então
    #    uma abertura SUAVE (3×3) remove ruído sem destruir a vegetação real.
    k7e = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    k3e = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k7e, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  k3e, iterations=1)

    # 4. Preenche buracos internos (vazios cercados por vegetação)
    mask = ndimage.binary_fill_holes(mask).astype(np.uint8)

    # 5. Remove componentes conexos menores que min_area_frac da imagem
    n, lbl, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    min_area = min_area_frac * mask.size
    limpa = np.zeros_like(mask)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            limpa[lbl == i] = 1
    return limpa

k7 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))

def exg_ingenuo(img_rgb):
    """Método histórico (ExG > 0 + morfologia) — mantido para comparação."""
    f = img_rgb.astype(np.float32) / 255.0
    R, G, B = f[:, :, 0], f[:, :, 1], f[:, :, 2]
    m = (2*G - R - B > 0).astype(np.uint8)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN,  k7, iterations=2)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k7, iterations=2)
    return m

# Roda histórico (ExG>0) e final (ExGR+HSV) nas DUAS imagens, guardando para a figura
resultados_seg = {}
print(f'{"Imagem (drone)":<24}{"ExG>0 (historico)":>18}{"ExGR+HSV (final)":>18}')
print('-' * 60)
for nome in ['drone-campo-fazenda', 'drone-lavoura-solo']:
    im = cv2.cvtColor(cv2.imread(f'../images/{nome}.jpg'), cv2.COLOR_BGR2RGB)
    m_old = exg_ingenuo(im)
    m_new = segmentar_aprimorado(im)
    resultados_seg[nome] = {'img': im, 'antigo': m_old, 'novo': m_new}
    print(f'{nome:<24}{m_old.mean()*100:>17.1f}%{m_new.mean()*100:>17.1f}%')

print('\nNa lavoura-solo, o fechamento-primeiro une as fileiras da cultura: a vegetação')
print('verde real (~45%) deixa de ser apagada — e o solo marrom continua excluído (ExGR<0).')

In [ ]:
# ── Figura "histórico → final" para as DUAS imagens ───────────────────────────
# Coluna 2 preserva o resultado antigo (registro); colunas 3-4 mostram a resolução.
fig, ax = plt.subplots(2, 4, figsize=(24, 11))

for row, (nome, r) in enumerate(resultados_seg.items()):
    im, m_old, m_new = r['img'], r['antigo'], r['novo']

    ax[row, 0].imshow(im)
    ax[row, 0].set_title(f'{nome}\nOriginal', fontsize=11)

    ax[row, 1].imshow(m_old, cmap='gray')
    ax[row, 1].set_title(f'HISTÓRICO — ExG>0  ({m_old.mean()*100:.1f}%)', fontsize=11)

    ax[row, 2].imshow(m_new, cmap='gray')
    ax[row, 2].set_title(f'FINAL — ExGR+HSV  ({m_new.mean()*100:.1f}%)', fontsize=11)

    # Destaque por dessaturação da máscara final (contorno mais grosso p/ alta resolução)
    ax[row, 3].imshow(destacar(im, m_new.astype(bool), espessura=8))
    ax[row, 3].set_title('FINAL — vegetação destacada', fontsize=11)

for a in ax.flat:
    a.axis('off')

plt.suptitle('Etapa 13 — Histórico → Final: o falso positivo do solo é resolvido '
             '(lavoura: 97% → valor real)', fontsize=14, y=1.0)
plt.tight_layout()
plt.savefig('../outputs/segmentacao_aprimorada.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/segmentacao_aprimorada.png')

## Etapa 14 — Generalização: a mesma metodologia no setor de petróleo

A receita do projeto **não depende do verde**. Seus blocos — assinatura espectral + testes
absolutos + morfologia + contornos + área % — segmentam qualquer alvo com assinatura de cor
distinta do fundo. Demonstração com um caso real do **setor de petróleo**:

**Imagem:** derramamento da *Deepwater Horizon* (Golfo do México), sensor **MODIS/Terra da
NASA**, 24 de maio de 2010 — domínio público.
Fonte: NASA Earth Observatory, via Wikimedia Commons
(`File:Deepwater Horizon oil spill - May 24, 2010.jpg`).

**Melhorando a acurácia (a mesma lição da vegetação).** A 1ª tentativa usou só **brilho**
(`S<60 & V>140`) — mas **nuvens** também são claras e dessaturadas e entravam como falso
positivo. A assinatura *real* do óleo é a **cor quente (tan)**: no *sunglint* (reflexo solar),
o petróleo + dispersante deixam a superfície com **R nitidamente maior que B**.

| Superfície | R − B (temperatura) | Decisão |
|---|---|---|
| **Óleo (tan)** | **+20 a +25** | aceito |
| Água | −5 a +8 | rejeitado |
| Nuvem branca | ≈ 0 | rejeitado |

Critério acurado: `R−B ≥ 14 & V > 110 & S < 110` + o mesmo refino morfológico. Assim como na
vegetação **a cor venceu o brilho** — aqui também: trocar brilho por cor quente elimina as
nuvens e delimita a mancha como uma região coerente.

**Aplicações no setor de petróleo:**

| Contexto | Como a metodologia se aplica |
|---|---|
| **Monitoramento de derrames** | Contornos + área % da mancha em imagens de satélite/aéreas — exatamente este experimento |
| **Faixa de dutos (right-of-way)** | Contornos da vegetação invadindo a faixa de servidão — manutenção preventiva |
| **Detecção indireta de vazamentos** | Queda de ExG/VARI na vegetação ao longo da linha denuncia solo contaminado antes do afloramento |

**Limitação honesta:** sem bandas térmicas/radar, água costeira com sedimento pode confundir-se
com óleo fino — em operação real, esta detecção RGB serve de **triagem rápida** antes de
sensores especializados (SAR).

In [ ]:
# ── Etapa 14: a MESMA receita, outro alvo (com acurácia melhorada) ────────────
# Imagem NASA/MODIS Terra, 24/mai/2010 — derramamento da Deepwater Horizon,
# Golfo do México. Domínio público (NASA). Fonte documentada no relatório.
img_nasa_full = cv2.cvtColor(cv2.imread('../images/nasa-deepwater-horizon.jpg'),
                             cv2.COLOR_BGR2RGB)
hn = int(img_nasa_full.shape[0] * 1600 / img_nasa_full.shape[1])
img_nasa = cv2.resize(img_nasa_full, (1600, hn), interpolation=cv2.INTER_AREA)

hsv_n = cv2.cvtColor(img_nasa, cv2.COLOR_RGB2HSV)
Sn, Vn = hsv_n[:, :, 1], hsv_n[:, :, 2]
Rn = img_nasa[:, :, 0].astype(np.int16)
Bn = img_nasa[:, :, 2].astype(np.int16)
warm = Rn - Bn   # temperatura de cor: positivo = tom quente (tan/marrom do óleo)

def refinar_oleo(m, frac):
    """Mesmo refino da vegetação: abertura + fechamento + buracos + área mínima."""
    m = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_OPEN,  k7, iterations=2)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k7, iterations=2)
    m = ndimage.binary_fill_holes(m).astype(np.uint8)
    n, lbl, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    out = np.zeros_like(m)
    for i in range(1, n):
        if stats[i, cv2.CC_STAT_AREA] >= frac * m.size:
            out[lbl == i] = 1
    return out

# 1ª tentativa (HISTÓRICO): só BRILHO (S<60 & V>140). Problema: NUVENS também são
# claras e dessaturadas → entravam como falso positivo (baixa acurácia).
mask_v1 = refinar_oleo((Sn < 60) & (Vn > 140), 0.001)

# Versão ACURADA: a assinatura real do óleo é a COR QUENTE (tan), não o brilho.
#   óleo  → warm = R-B ≈ 20-25 (petróleo + dispersante)
#   água  → warm ≈ -5 a 8 (neutra/azulada)  → excluída
#   nuvem → warm ≈ 0 (branca neutra)         → excluída
# É a MESMA lição da vegetação: a cor distingue o alvo, o brilho engana.
mask_final = refinar_oleo((warm >= 14) & (Vn > 110) & (Sn < 110), 0.0012)

pct_v1   = mask_v1.mean() * 100
pct_oleo = mask_final.mean() * 100
cont_oleo, _ = cv2.findContours(mask_final, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

print(f'HISTORICO (so brilho, S<60 & V>140):  {pct_v1:>5.1f}%  (incluia nuvens)')
print(f'ACURADO   (cor quente, R-B>=14):      {pct_oleo:>5.1f}%  em {len(cont_oleo)} regiao(oes)')
print()
print('Mapeamento da metodologia (vegetacao -> oleo):')
print('  assinatura de cor          verde dominante (ExG)  ->  tom quente R-B (tan)')
print('  testes absolutos           ExG>0 & trava HSV      ->  R-B>=14 & V>110 & S<110')
print('  morfologia+area+contornos  identico')

fig, ax = plt.subplots(1, 4, figsize=(24, 5.5))
ax[0].imshow(img_nasa)
ax[0].set_title('NASA/MODIS — Deepwater Horizon\n24 mai 2010', fontsize=11)
ax[1].imshow(mask_v1, cmap='gray')
ax[1].set_title(f'HISTÓRICO — só brilho  ({pct_v1:.1f}%)\nnuvens entram (falso positivo)', fontsize=11)
ax[2].imshow(mask_final, cmap='gray')
ax[2].set_title(f'ACURADO — cor quente R-B  ({pct_oleo:.1f}%)\nnuvens e água excluídas', fontsize=11)
ax[3].imshow(destacar(img_nasa, mask_final.astype(bool), escurecer=0.45, espessura=3))
ax[3].set_title('Mancha destacada + contornos', fontsize=11)
for a in ax:
    a.axis('off')
plt.suptitle('Etapa 14 — Generalização ao setor de petróleo: detecção por COR (não por brilho)',
             fontsize=14, y=1.04)
plt.tight_layout()
plt.savefig('../outputs/generalizacao_petroleo.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/generalizacao_petroleo.png')

## Etapa 15 — Generalização à astronomia: detectar e contar objetos

A ideia inicial do projeto era trabalhar com **imagens de telescópio**. Aqui ela fecha a
demonstração: a mesma metodologia que mede *cobertura* (vegetação) e *área* (óleo) também
**conta objetos**. Muda só a assinatura.

**Imagem:** aglomerado globular **M13**, Telescópio Espacial **Hubble (NASA/ESA)** — domínio
público (`File:Hubble image of globular cluster M13 (opo0840a).jpg`).

**Assinatura:** estrela = **ponto claro sobre céu negro** — aqui é o **brilho**, não a cor. A
versão final combina **duas peças que já apareceram no projeto**:

1. **White top-hat** — o realce de contraste *local* (parente da equalização da Etapa 9c) que
   faz as estrelas fracas saltarem do fundo, vizinhança a vizinhança. O limiar global de brilho
   perdia essas estrelas; o top-hat as recupera.
2. **Otsu** — o *mesmo* limiar automático que segmentou por brilho na Etapa 10 escolhe o corte,
   **sem número mágico**. Em seguida, o `cv2.connectedComponentsWithStats` — *a mesma* função do
   filtro de área da vegetação e do óleo — **conta** cada componente conexo como uma estrela.

**Resultado:** a contagem quase **dobra** em relação ao limiar global (de ~7,7 mil para ~14,6 mil
estrelas). O zoom da periferia mostra a detecção estrela a estrela (círculos verdes).

**Limitação honesta:** no núcleo denso do aglomerado, estrelas vizinhas se fundem em um único
blob — a contagem é um **limite inferior**. É a mesma natureza das limitações já vistas
(sedimento no óleo, vegetação seca na lavoura): *toda detecção por limiar tem um regime onde
o sinal e o fundo se confundem.*

> **Fecho da ideia central:** os mesmos blocos — assinatura (cor *ou* brilho) → limiar absoluto
> → morfologia → componentes conexos → medida — mediram **% de vegetação**, **área de mancha de
> óleo** e **contagem de estrelas**. A contribuição do trabalho é a **metodologia**, não um
> detector de verde.

In [ ]:
# ── Etapa 15: detecção e CONTAGEM de objetos — astronomia ─────────────────────
# Imagem Hubble (NASA/ESA) do aglomerado globular M13. Domínio público.
# Aqui a assinatura NÃO é a cor, e sim o BRILHO: estrela = ponto claro no céu negro.
img_m13_full = cv2.cvtColor(cv2.imread('../images/hubble-m13-cluster.jpg'),
                            cv2.COLOR_BGR2RGB)
hm = int(img_m13_full.shape[0] * 1400 / img_m13_full.shape[1])
img_m13 = cv2.resize(img_m13_full, (1400, hm), interpolation=cv2.INTER_AREA)
gray_m13 = cv2.cvtColor(img_m13, cv2.COLOR_RGB2GRAY)

def contar_estrelas(mask):
    """Conta estrelas: componentes conexos com área de estrela (2–400 px)."""
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((2, 2), np.uint8))
    n, lbl, stats, cen = cv2.connectedComponentsWithStats(mask, connectivity=8)
    idx = [i for i in range(1, n) if 2 <= stats[i, cv2.CC_STAT_AREA] <= 400]
    return idx, cen, mask

# 1ª versão (histórico): threshold global de brilho > 140. Problema: as estrelas
# FRACAS (a maioria!) ficam abaixo do corte e não eram contadas.
est_v1, _, _ = contar_estrelas(gray_m13 > 140)

# Versão FINAL — duas peças que já usamos no projeto, reaproveitadas:
#  (1) WHITE TOP-HAT: realce de contraste LOCAL (o parente local da equalização da Etapa 9c),
#      que faz as estrelas fracas saltarem do fundo vizinhança a vizinhança;
#  (2) OTSU: o MESMO limiar automático da segmentação por brilho (Etapa 10) escolhe o corte,
#      sem número mágico. Depois, o MESMO connectedComponents CONTA cada estrela.
tophat = cv2.morphologyEx(gray_m13, cv2.MORPH_TOPHAT,
                          cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
thr_estrela, binar = cv2.threshold(tophat, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
estrelas, cen, mask_estrelas = contar_estrelas(binar > 0)

print(f'HISTORICO (brilho global > 140):           {len(est_v1):>6} estrelas (perdia as fracas)')
print(f'FINAL (top-hat + Otsu, corte automatico={thr_estrela:.0f}): {len(estrelas):>6} estrelas')
print('Limitacao honesta: no nucleo denso, estrelas se fundem em blobs —')
print('a contagem e um limite INFERIOR (analogo a agua+sedimento no oleo).')

# Zoom na periferia para evidenciar a detecção estrela a estrela
y0, y1, x0, x1 = 120, 520, 900, 1300
zoom = img_m13[y0:y1, x0:x1].copy()
n_zoom = 0
for i in estrelas:
    cx, cy = cen[i]
    if x0 <= cx < x1 and y0 <= cy < y1:
        cv2.circle(zoom, (int(cx - x0), int(cy - y0)), 5, (0, 255, 120), 1)
        n_zoom += 1

fig, ax = plt.subplots(1, 3, figsize=(21, 7))
ax[0].imshow(img_m13)
ax[0].set_title('Hubble M13 (NASA/ESA) — original', fontsize=12)
ax[1].imshow(mask_estrelas, cmap='gray')
ax[1].add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='#D97757', lw=2))
ax[1].set_title(f'Top-hat + Otsu → {len(estrelas)} estrelas contadas\n(antes só {len(est_v1)} por brilho global)', fontsize=12)
ax[2].imshow(zoom)
ax[2].set_title(f'Zoom da periferia: {n_zoom} estrelas detectadas (círculos)', fontsize=12)
for a in ax:
    a.axis('off')
plt.suptitle('Etapa 15 — Astronomia: contar objetos com connectedComponents (o mesmo do óleo)',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../outputs/generalizacao_estrelas.png', dpi=130, bbox_inches='tight')
plt.show()
print('Figura salva em outputs/generalizacao_estrelas.png')